# Ask 4 — Meaning as Numbers

An embedding model maps text to a vector; similar meanings land nearby.
First we prove that fancier *word* math still can't cross the synonym gap
(offline), then the real model crosses it (in Colab, free, no key).

Run in Colab for the full experience — the model download is ~90 MB, no
API key involved, and your pile never leaves the notebook at this stage.

In [ ]:
# The pile: documents from the (fictional) Jefferson High School.
# Real enough to search, small enough to read whole.
PILE = {
 "handbook_academics": """S4.1 Grading scale. A 90-100, B 80-89, C 70-79, D 60-69.
Semester grades weight exams at 30 percent.
S4.2 Exam Retake Policy. This policy applies to final exams only. Students
receive one retake per semester, requested within ten school days. The
higher score stands.
S4.3 Grade appeals. Appeals go to the department head in writing within
fifteen school days of the posted grade.
S4.5 Late work. Assignments lose 10 percent per school day late, to a
maximum of 50 percent. Teachers may grant extensions for documented
emergencies.""",
 "handbook_schedule": """S2.0 Bell schedule. Regular days run eight periods,
8:15 AM to 3:20 PM.
S2.1 Wednesday schedule. Dismissal at 1:30 PM every Wednesday for staff
development.
S2.4 Late arrival. Students arriving after 8:30 AM sign in at the main
office with a note.""",
 "handbook_trips": """S5.1 Field trips require a signed permission form
submitted five school days in advance.
S5.2 Trip costs above 20 dollars qualify for the student activity fund.
S5.4 Chaperones must be approved district volunteers.""",
 "handbook_athletics": """S6.2 Eligibility. Athletes must hold a C average
during their season. Freshmen may try out for varsity teams.
S6.3 Petitions. A varsity roster spot for a freshman requires a coach's
petition to the athletic director.""",
 "robotics_minutes": """Robotics club meets Tuesdays in room 214. Regional
trip is April 18; bring your signed permission form by April 10. Dues are
15 dollars for the year.""",
 "clubs_list": """Active clubs: robotics (Tuesdays), debate (Thursdays),
art collective (Fridays), chess (lunch, library). Sign-up forms at the
student office.""",
 "bus_routes": """Routes 12 and 15 serve the north side. Final pickup at
4:45 PM outside door C. Activity buses run Tuesday and Thursday only.""",
 "cafeteria": """Lunch periods run 11:10, 11:55, and 12:40. Breakfast is
served from 7:40 AM. Menus post monthly on the food services page.""",
}
print(f"{len(PILE)} documents, {sum(len(t) for t in PILE.values())} characters total")

In [ ]:
def chunk_by_section(pile, overlap_sentences=1):
    """Cut on the S-section seams; carry a sentence of overlap across cuts."""
    chunks = []
    for doc, text in pile.items():
        parts, current, header = [], [], None
        for line in text.splitlines():
            if line.strip().startswith("S") and len(line) > 2 and line.strip()[1].isdigit():
                if current:
                    parts.append((header, " ".join(current)))
                header, current = line.strip().split()[0].rstrip("."), [line]
            else:
                current.append(line)
        if current:
            parts.append((header, " ".join(current)))
        for i, (header, body) in enumerate(parts):
            text_out = body
            if overlap_sentences and i > 0:
                prev_tail = parts[i-1][1].split(". ")[-1]
                text_out = prev_tail + " ... " + body
            chunks.append({"doc": doc, "section": header or doc, "text": " ".join(text_out.split())})
    return chunks

CHUNKS = chunk_by_section(PILE)
print(f"{len(CHUNKS)} chunks")
for c in CHUNKS[:3]:
    print(f"  [{c['doc']} {c['section']}] {c['text'][:70]}...")

In [ ]:
import math, re, collections

def words(text):
    return [w for w in re.findall(r"[a-z0-9]+", text.lower()) if len(w) > 2]

# document frequency: in how many chunks does each word appear?
DF = collections.Counter()
for c in CHUNKS:
    for w in set(words(c["text"])):
        DF[w] += 1

def score(query, chunk):
    """Shared words, each weighted by rarity: rare words shout, common words whisper."""
    shared = set(words(query)) & set(words(chunk["text"]))
    return sum(1.0 / DF[w] for w in shared)

def retrieve(query, k=3):
    ranked = sorted(CHUNKS, key=lambda c: -score(query, c))
    return ranked[:k]

print("scorer ready")

## Word-count cosine — the control experiment

Cosine similarity over word-count vectors is real vector math — and it is
still word matching wearing a gown. If vectors alone were the trick, this
would fix lesson 3's miss. It doesn't:

In [ ]:
import math, collections

def wordvec(text):
    return collections.Counter(words(text))

def cosine(v1, v2):
    shared = set(v1) & set(v2)
    dot = sum(v1[w] * v2[w] for w in shared)
    n1 = math.sqrt(sum(x*x for x in v1.values()))
    n2 = math.sqrt(sum(x*x for x in v2.values()))
    return dot / (n1 * n2) if n1 and n2 else 0.0

SYNONYM_Q = "when do we get to leave early midweek"
qv = wordvec(SYNONYM_Q)
ranked = sorted(CHUNKS, key=lambda c: -cosine(qv, wordvec(c["text"])))
answer_rank = [c["section"] for c in ranked].index("S2.1") + 1
print(f"word-count cosine ranks the answer chunk #{answer_rank}")
print("Vectors, yes. Meaning, no - the numbers only count spelling.")
assert answer_rank > 1, "word-count cosine should still miss the synonym question"

## The real thing — a trained embedding model

The cell below downloads a small open model (`all-MiniLM-L6-v2`) and
embeds every chunk. It learned from billions of sentences that "get out
early" and "dismissal" live in the same contexts — that knowledge is what
the 384 numbers per text encode.

**What to expect when you run it:** the synonym question's nearest chunk
becomes S2.1 (the dismissal section), typically by a wide cosine margin,
and the lesson-3 question set improves — measure it with your own eyes in
the last cell; exact scores vary slightly by model version.

In [ ]:
%pip install -q sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")   # ~90 MB, free, local
chunk_vecs = model.encode([c["text"] for c in CHUNKS])
print("embedded", len(chunk_vecs), "chunks; each is", chunk_vecs.shape[1], "numbers")

def retrieve_meaning(query, k=3):
    qv = model.encode([query])[0]
    sims = chunk_vecs @ qv / (np.linalg.norm(chunk_vecs, axis=1) * np.linalg.norm(qv))
    order = np.argsort(-sims)
    return [(float(sims[i]), CHUNKS[i]) for i in order[:k]]

for sim, c in retrieve_meaning("when do we get to leave early midweek"):
    print(f"{sim:.3f}  [{c['doc']} {c['section']}]  {c['text'][:60]}...")

## The before/after table

In [ ]:
QUESTIONS = [
    ("How many final exam retakes do I get?", "S4.2"),
    ("What is the late work penalty?", "S4.5"),
    ("When are field trip permission forms due?", "S5.1"),
    ("When does robotics club meet?", "robotics_minutes"),
    ("What time is Wednesday dismissal?", "S2.1"),
    ("when do we get to leave early midweek", "S2.1"),
    ("can we bail before the end of the day midweek", "S2.1"),
    ("Do ninth graders ever play varsity?", "S6.2"),
    ("How do I fight a bad grade?", "S4.3"),
]

print(f"{'question':45}  words  meaning")
for q, want in QUESTIONS:
    w = any(c["section"] == want or c["doc"] == want for c in retrieve(q))
    m = any(c["section"] == want or c["doc"] == want for _s, c in retrieve_meaning(q))
    print(f"{q[:45]:45}  {'HIT ' if w else 'miss'}   {'HIT ' if m else 'miss'}")
print()
print("Expect the synonym rows to flip to HIT. If one didn't, you just found")
print("your first lesson-7 case - keep it.")

## Try it

1. Embed your own ten-question set and build this table for your pile.
2. Find a question where MEANING search does worse than word search (they
   exist — often when a rare exact term like a form number is the whole
   signal). What does that suggest about combining the two?
3. **Build turn-in:** your before/after table and three diagnosis
   sentences: what flipped, what didn't, what both still miss.